## **Fine tuning Bert for Phising URL Identification** 

## Imports 

In [15]:
!pip install evaluate


## Load Data

In [43]:
from datasets import DatasetDict, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
from transformers import DataCollatorWithPadding  # Corrected typo


In [44]:
from datasets import load_dataset


  # This will prompt for your Hugging Face token
dataset = load_dataset("shawhin/phishing-site-classification")


In [45]:
dataset['train'][0] 

{'text': "http://bazurashop.com/idex.html?sfm_from_iframe=1',300,350",
 'labels': 1}

## Load Pretained model 

In [46]:
# define pre trained model path 
model_path = "google-bert/bert-base-uncased" 
#  load model tokeizer 
tokenizer  = AutoTokenizer.from_pretrained(model_path) 
id2label =  {0: "Safe" , 1: "Not Safe"}  
label2id = {"Safe": 0 , "Not Safe" : 1 } 
model =  AutoModelForSequenceClassification.from_pretrained(model_path, num_labels =2,
                                                           id2label = id2label ,
                                                           label2id=label2id) 

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Set trainable perameters 


In [47]:
# freeze all base model parameters 
for name , param in model.base_model.named_parameters() : 
    param.requires_grad = False  
# freeze base model pooling layers 
for name, param in model.base_model.named_parameters(): 
    if "pooler" in name :
        param.require_grad =  True 

## Data Processing 

In [48]:
def preprocess_function(examples): 
    # return tokenized text with trucntion 
    return tokenizer(examples["text"] , truncation = True) 

tokenized_data =  dataset.map(preprocess_function, batched = True)

In [49]:
data_collator =  DataCollatorWithPadding(tokenizer = tokenizer) 

In [50]:
import evaluate
import numpy as np
from scipy.special import softmax

# Load metrics correctly
accuracy = evaluate.load("accuracy")
auc_score = evaluate.load("roc_auc")  # Fix the typo here

def compute_metrics(eval_pred):
    # Get predictions
    predictions, labels = eval_pred
    
    # Apply softmax to get probabilities
    probabilities = softmax(predictions, axis=-1)  # Fix here
    
    # Extract probabilities for the positive class
    positive_class_probs = probabilities[:, 1]  # Fix typo (probalisties -> probabilities)
    
    # Compute AUC
    auc = np.round(auc_score.compute(prediction_scores=positive_class_probs, references=labels)["roc_auc"], 3)
    
    # Compute accuracy
    predicted_classes = np.argmax(predictions, axis=1)
    acc = np.round(accuracy.compute(predictions=predicted_classes, references=labels)["accuracy"], 3)
    
    return {"Accuracy": acc, "AUC": auc}



## training perameters 

In [51]:
lr = 2e-4
batch_size = 8
num_epochs = 10

training_args = TrainingArguments(
    output_dir="bert-phishing-classifier_teacher",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)


In [57]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset,
    tokenizer=tokenizer,  # 🔴 Deprecated in v5.0.0
    data_collator=tokenizer
) 

<ipython-input-57-609ec52af176>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## Apply Model to Validation Dataset 

In [ ]:
predictions = trainer.predict(tokenized_data["validation"])

# Extract the logits and labels from the predictions object
logits = predictions.predictions
labels = predictions.label_ids

# Use your compute_metrics function
metrics = compute_metrics((logits, labels))
print(metrics) 

In [62]:
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model and tokenizer
model_path = "./bert-phishing-model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Move model to the correct device
model.to(device)

# Example URLs for prediction
urls = ["http://secure-bank-login.com", "http://malicious-phishing-site.com"]

# Tokenize the URLs and move inputs to the same device as the model
inputs = tokenizer(urls, padding=True, truncation=True, return_tensors="pt").to(device)

# Make predictions
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# Get logits and convert to class labels
logits = outputs.logits
predictions = torch.argmax(logits, dim=-1)

# Define label mapping
id2label = {0: "Safe", 1: "Not Safe"}

# Print predictions
for url, pred in zip(urls, predictions.cpu().numpy()):  # Move to CPU for printing
    print(f"URL: {url} -> Prediction: {id2label[pred]}")


OSError: Incorrect path_or_model_id: './bert-phishing-model'. Please provide either the path to a local folder or the repo_id of a model on the Hub.